**RMSprop (Root Mean Square Propagation)** is an adaptive learning rate optimization algorithm designed to solve a major flaw in earlier optimizers by dynamically scaling the step size for every individual parameter.

---

### Definition

**RMSprop** is an optimization method that maintains a moving (exponentially decaying) average of squared gradients for each weight. It divides the learning rate by the square root of this average, allowing the model to take large steps on gentle slopes and small steps on steep, oscillating slopes.

---

### The Problem RMSprop Solved (Before AdaGrad)

To understand RMSprop, you have to understand what came right before it: **AdaGrad** (Adaptive Gradient Algorithm).

* AdaGrad was revolutionary because it gave every parameter its own learning rate based on historical gradients.
* **The Flaw:** AdaGrad accumulated *every past squared gradient* from the very first epoch to the current one. Because squares are always positive, this running sum grew larger and larger indefinitely. As a result, the effective learning rate shrank closer and closer to zero, causing training to grind to a complete halt prematurely.

**RMSprop was invented by Geoffrey Hinton** specifically to fix this flaw by replacing the cumulative sum with an **exponentially decaying moving average**, giving more weight to *recent* gradients and forgetting ancient history.

---

### How RMSprop Works (The Mechanism)

At every training step $t$, RMSprop performs three key calculations:

1. **Compute the Gradient ($g_t$):** Calculate the gradient of the loss function for the current mini-batch.
2. **Update the Moving Average of Squared Gradients ($v_t$):**

$$v_t = \beta v_{t-1} + (1 - \beta) g_t^2$$


* Where $\beta$ is the decay/forgetting factor (typically set to $0.9$). This ensures that old gradients fade away, preventing the denominator from growing infinitely.


3. **Update the Weights ($w_t$):**

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{v_t} + \epsilon} \odot g_t$$


* Where $\eta$ is the base learning rate, $\epsilon$ (epsilon) is a tiny number (e.g., $10^{-8}$) to prevent division by zero, and $\odot$ denotes element-wise multiplication.



#### Why This Works Brilliantly:

* If a parameter experiences **consistently large, erratic gradients** (steep walls), $v_t$ becomes large. Dividing by a large number ($\sqrt{v_t}$) shrinks the effective learning rate, calming down the oscillations.
* If a parameter experiences **small, consistent gradients** (gentle slopes), $v_t$ stays small. Dividing by a small number boosts the effective learning rate, accelerating progress across plateaus.

---

### Where Is RMSprop Used Today?

* **Recurrent Neural Networks (RNNs & LSTMs):** Historically, RMSprop was the go-to optimizer for sequence models because it handles non-stationary gradients in recurrent connections exceptionally well.
* **Reinforcement Learning (RL):** It remains a favorite optimizer for deep reinforcement learning algorithms (like DQN and PPO) because reward landscapes in RL change dynamically as the agent learns.
* **The Ancestor of Adam:** RMSprop is fundamentally important because **Adam is essentially RMSprop combined with Momentum**. Adam took RMSprop's second moment (variance tracking) and paired it with SGD's first moment (momentum tracking).

---

### PyTorch Implementation

Here is how you initialize and use RMSprop in PyTorch:

```python
import torch
import torch.optim as optim

# Assuming `model` is already defined
optimizer = optim.RMSprop(
    model.parameters(), 
    lr=0.001,          # Base learning rate
    alpha=0.9,         # Smoothing constant (decay rate for moving average, equivalent to beta)
    weight_decay=1e-4  # L2 regularization
)

```